In [1]:
import re
import json
import pandas as pd

In [13]:
path = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
train_df = pd.read_parquet(path + 'train_set.parquet', engine='fastparquet')
test_df = pd.read_parquet(path + 'test_set.parquet', engine='fastparquet')
eval_df = pd.read_parquet(path + 'eval_set.parquet', engine='fastparquet')

In [2]:
english_stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll",
                     "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's",
                     'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs',
                     'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am',
                     'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does',
                     'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while',
                     'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
                     'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off',
                     'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
                     'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor',
                     'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don',
                     "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren',
                     "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn',
                     "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't",
                     'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't",
                     'won', "won't", 'wouldn', "wouldn't", '\\.', '\\?', ',', '\\!', "'s", '']


def clean_stopwords(sample):
    """
    :param sample: List[Str], lower case
    :return:  List[Str]
    """
    return [token for token in sample if token not in english_stopwords]

In [8]:
def clean_str(string):
    """
    Original Source:  https://github.com/yoonkim/CNN_sentence/blob/master/process_data.py
    :param string: Str
    :return -> Str
    """
    string = string.strip().strip('"')
    # Added 'r' before the pattern strings
    string = re.sub(r"[^A-Za-z(),!?\.\'\`]", " ", string)
    string = re.sub(r"\'s", " \'s", string)
    string = re.sub(r"\'ve", " \'ve", string)
    string = re.sub(r"n\'t", " n\'t", string)
    string = re.sub(r"\'re", " \'re", string)
    string = re.sub(r"\'d", " \'d", string)
    string = re.sub(r"\'ll", " \'ll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"\.", " . ", string) # Note: literal dot doesn't always need escape in replacement
    string = re.sub(r"\"", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " ( ", string)
    string = re.sub(r"\)", " ) ", string)
    string = re.sub(r"\?", " ? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    return string.strip().lower()

In [104]:
def preprocess_line(sample):
    """
    :param sample: Str, "The sample would be tokenized and filtered according to the stopwords list"
    :return: token_list -> List[Str]
    """
    sample = clean_str(sample.lstrip().rstrip())
    token_list = clean_stopwords(sample.split(' '))
    # return json.dumps({'token': token_list, 'label': []})
    return {'token': token_list, 'label': []}

In [124]:
def preprocess_df_text(df):
    """
    :param df: DataFrame, the entire DataFrame
    :return: List[Dict{'token': List[Str], 'label': []}]
    """
    corpus_data = list()
    raw_data = list()
    col_names = ['DOI','Title','Abstract']
    for _, row in df.iterrows():
        line = row['Title'] + row['Abstract']
        sample_tokens = preprocess_line(line)
        labels_df = row.drop(columns=col_names)
        labels_ls = list(labels_df[labels_df == 1].index)
        raw_data.append({'token': line.rstrip(), 'label': labels_ls})
        # corpus_data.append(json.dumps({'token': sample_tokens, 'label': []}))
        sample_tokens['label'] = labels_ls
        corpus_data.append(sample_tokens)
    return raw_data, corpus_data

In [127]:
def save_processed_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

In [131]:
def load_processed_file(file_path):
    """
    :param file_path: Str, file path of the processed file
    :return: List[Dict{'token': List[Str], 'label': []}]
    """
    loaded_data = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # Load each line individually and append to our list
            loaded_data.append(json.loads(line))
    return loaded_data

In [134]:
train_raw, train_corpus = preprocess_df_text(train_df)
test_raw, test_corpus = preprocess_df_text(test_df)
eval_raw, eval_corpus = preprocess_df_text(eval_df)

In [135]:
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/HiAGM/'
save_processed_file(train_raw, filepath + 'train_raw.json')
save_processed_file(test_raw, filepath + 'test_raw.json')
save_processed_file(eval_raw, filepath + 'eval_raw.json')

save_processed_file(train_corpus, filepath + 'train_corpus.json')
save_processed_file(test_corpus, filepath + 'test_corpus.json')
save_processed_file(eval_corpus, filepath + 'eval_corpus.json')

# Hierarchy

In [2]:
# Load the ontology
from owlready2 import *
ontology_path='/Users/fdp54928/Documents/PaNET-classifier/Data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

In [3]:
# Run reasoner

with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit:/Users/fdp54928/Documents/PaNET-classifier/.venv/lib/python3.13/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jt/qsn_d43943dbh0h9ngcrb3n80000gq/T/tmp0nopbx8g
* Owlready2 * HermiT took 1.1049606800079346 seconds
* Owlready * Reparenting PaNET.PaNET1001000: {owl.ObjectProperty, owl.IrreflexiveProperty, owl.AsymmetricProperty, PaNET.PaNET1000000} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1004000: {owl.ObjectProperty, owl.IrreflexiveProperty, owl.AsymmetricProperty, PaNET.PaNET1000000} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1002000: {owl.ObjectProperty, owl.IrreflexiveProperty, owl.AsymmetricProperty, PaNET.PaNET1000000} => {PaNET.PaNET1000000}
* Owlready * Reparenting PaNET.PaNET1003000: {owl.ObjectProperty, owl.IrreflexiveProperty, owl

In [ ]:
def add_classes_recursively(cls, subclass_map):

    excluded_iris = { 
    'http://www.w3.org/2002/07/owl#Thing',
    'https://www.wikidata.org/wiki/Q133900'
    }

    if cls.iri not in excluded_iris:
        ls = [subclass.iri for subclass in cls.subclasses()]
        subclass_map[cls.iri] = ls
        for subclass in cls.subclasses():
            add_classes_recursively(subclass, subclass_map)
    return subclass_map



root = onto.search_one(iri='http://purl.org/pan-science/PaNET/PaNET00001')
subclass_map = {}

subclass_map = add_classes_recursively(root, subclass_map)

In [14]:
import pickle
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/'
with open(filepath + 'subclass_map.pkl', 'wb') as f:
    pickle.dump(subclass_map, f)

In [15]:
# Load the subclass map
import joblib
filepath = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/subclass_map.pkl'
subclass_map = joblib.load(filepath)

In [17]:
path = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/HiAGM/'
with open(f"{path}panet_pair.taxonomy", 'w', encoding='utf-8') as f:
    for parent, children in subclass_map.items():
        for child in children:
            # Write: Parent [TAB] Child [NEWLINE]
            f.write(f"{parent}\t{child}\n")

In [18]:
path = '/Users/fdp54928/Documents/PaNET-classifier/Data/training_testing_data/HiAGM/'
with open(f"{path}panet.taxonomy", 'w', encoding='utf-8') as f:    
    for parent, children in subclass_map.items():
        # Join the parent and all children with tabs
        line = "\t".join([parent] + children)
        f.write(line + "\n")